# MNIST 次元削減の復元精度・プライバシー攻撃 実験

このノートブックでは、`dimensionality_reduction.py` と `anchor_utils.py` を用いて、
次元削減後の表現から元の特徴量をどの程度復元できるか、および将来的なプライバシー攻撃の実験用の枠組みを用意します。

基本設定は `base_config` で与えられた MNIST 用の設定を使用します。


In [1]:
import sys, os
from pathlib import Path
from types import SimpleNamespace

# リポジトリ直下や親ディレクトリから src を探索して import path に追加
cwd = Path.cwd()
candidate_roots = [cwd] + list(cwd.parents)
src_dir = None
for root in candidate_roots:
    cand = root / "src"
    if cand.exists():
        src_dir = cand
        break

if src_dir is None:
    raise RuntimeError(
        "src ディレクトリが見つかりません。"
        "ノートブックの実行ディレクトリをリポジトリ直下にするか、"
        "このセルのパス設定を修正してください。"
    )

if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

# ★ これを追加 ★
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
    
from pathlib import Path
import sys

repo_root = Path.cwd()
# ここが dca 直下でない場合は親を辿る
if not (repo_root / "src").exists():
    for p in repo_root.parents:
        if (p / "src").exists():
            repo_root = p
            break

# src とリポジトリ直下の両方を sys.path に追加
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))


# モジュールのインポート
from institution_data_pipeline.load_data import LOADERS
import dimensionality_reduction
from intermediate_expression import anchor_utils

print("OK: imported dimensionality_reduction and anchor_utils")
print("src_dir:", src_dir)
print("MNIST loader exists:", "mnist" in LOADERS)

OK: imported dimensionality_reduction and anchor_utils
src_dir: c:\Users\sueya\Git-Repositories\takano_labo\dca\src
MNIST loader exists: True


In [183]:
import numpy as np
import pandas as pd

base_config = {
    "name": "tutorial_mnist",
    "dataset": "mnist",
    "y_name": "target",
    "num_institution_user": 1200,
    "data_distribution": "even",
    "anchor_method": "smote",
    "num_anchor_data": 1200,
    "dim_intermediate": 600,
    "F_type": "svd",
    "seed": 42,
    "inter_normalization": False,
    "spill_num": 100,
}

cfg = SimpleNamespace(**base_config)

# MNIST DataFrame を読み込み
df_full = LOADERS[cfg.dataset]()
print("Full dataset shape:", df_full.shape)

# num_institution_user 行だけサンプリング
rng = np.random.default_rng(cfg.seed)
indices = rng.choice(len(df_full), size=cfg.num_institution_user, replace=False)
df_sub = df_full.iloc[indices].reset_index(drop=True)

y = df_sub[cfg.y_name].to_numpy()
X_raw = df_sub.drop(columns=[cfg.y_name]).to_numpy(dtype=np.float32)

from sklearn.preprocessing import StandardScaler
scaler_x = StandardScaler()
X = scaler_x.fit_transform(X_raw).astype(np.float32)

X.shape, y.shape


Full dataset shape: (70000, 785)


((1200, 784), (1200,))

In [184]:
# 次元削減の構築と適用
projector = dimensionality_reduction.build_dimensionality_projector(
    X,
    n_components=cfg.dim_intermediate,
    F_type=cfg.F_type,
    seed=int(cfg.seed),
    config=cfg,
)

X_tilde = projector(X)
X_tilde.shape


(1200, 600)

In [185]:
# アンカーデータ A の生成（anchor_utils.produce_anchor を利用）
num_row = cfg.num_anchor_data
num_col = X.shape[1]

anchor_cfg = SimpleNamespace(
    anchor_method=cfg.anchor_method,
    y_name=cfg.y_name,
    smote_ratio=1.0,
)

A = anchor_utils.produce_anchor(
    num_row=num_row,
    num_col=num_col,
    seed=int(cfg.seed),
    config=anchor_cfg,
    train_df=df_sub,
    Xs_train=[X],
    Xs_test=[X],
    ys_train=[y],
    ys_test=[y],
)

# 重要: X と同じ StandardScaler を A にも適用してから投影
A = scaler_x.transform(A).astype(np.float32)

A_tilde = projector(A)

A.shape, A_tilde.shape


((1200, 784), (1200, 600))

In [186]:
# アンカーの一部 A' / A~' が流出したと仮定
spill_num = int(cfg.spill_num)
spill_num = min(spill_num, A.shape[0])

idx_leak = rng.choice(A.shape[0], size=spill_num, replace=False)

A_leak = A[idx_leak]
A_tilde_leak = A_tilde[idx_leak]

A_leak.shape, A_tilde_leak.shape


((100, 784), (100, 600))

In [187]:
from sklearn.metrics import mean_squared_error

# 疑似逆行列ベースの f^{-1} の近似
# A~' * W ≈ A' となる W を最小二乗で推定し、x ≈ z W と復元する
Z = A_tilde_leak  # (spill_num, k)
X_target = A_leak  # (spill_num, d)

# lstsq で Z @ W ≈ X_target を解く
W_pinv, *_ = np.linalg.lstsq(Z, X_target, rcond=None)

def f_inv_pinv(Z_input: np.ndarray) -> np.ndarray:
    return Z_input @ W_pinv

# 元データ X に対する復元
X_rec_pinv = f_inv_pinv(X_tilde)

rmse_pinv = np.sqrt(mean_squared_error(X, X_rec_pinv))
rmse_pinv


0.5851254466135517

In [188]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# MLP による f^{-1} 近似（model.py の MLP 設定を参考）
scaler_z = StandardScaler()
Z_train = scaler_z.fit_transform(A_tilde_leak)

mlp = MLPRegressor(
    hidden_layer_sizes=(256,),
    activation="relu",
    solver="adam",
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=int(cfg.seed),
)

mlp.fit(Z_train, A_leak)

Z_all = scaler_z.transform(X_tilde)
X_rec_mlp = mlp.predict(Z_all)

rmse_mlp = np.sqrt(mean_squared_error(X, X_rec_mlp))
rmse_mlp


17.803308623143213

## 今後の拡張 (TODO)

- A に対して DP ノイズ（例: ガウス機密保持ノイズ）を付加してから次元削減し，
  上記の f^{-1} による復元精度を比較する。
- Attribute Inference / Membership Inference 攻撃を実装し，
  復元精度とプライバシー攻撃の成功率の関係を評価する。
